# EWS Fraud Detection - Tahap 2B: Ekstraksi Metrik Keuangan dari PDF

Notebook ini digunakan untuk mengekstrak metrik keuangan penting dari laporan keuangan (`Laporan_Keuangan`) yang terdaftar di `PDF_Inventory.xlsx`. 
Jika teks tidak dapat diekstrak secara langsung (karena dokumen hasil scan), skrip akan menggunakan OCR fallback (Tesseract).

Notebook ini dibagi menjadi dua bagian:
1. **Pipeline 1 (Original)**: Mengekstrak 9 metrik awal ke berkas `Financial_Metrics_Raw.xlsx`.
2. **Pipeline 2 (Focused & Deduplicated - Beneish Variables)**: Fokus ekstraksi variabel lengkap untuk perhitungan **Beneish M-Score** dengan fitur-fitur lanjutan:
   - **Deduplikasi Unik**: Memastikan hanya ada 1 baris per Perusahaan-Tahun dengan memprioritaskan dokumen `FinancialStatement` utama.
   - **Penyatuan Tanda Kurung Terpisah**: Menyatukan baris tanda kurung negatif `( )` yang terpisah akibat ekstraksi PyMuPDF ke angka di bawahnya.
   - **Spasi Horizontal Toleran**: Menggunakan RegEx spasi horizontal `[ \t]` untuk mencegah kebocoran/penggabungan angka antar tahun.
   - **Peningkatan Kedalaman Pindai**: Memindai hingga **40 halaman** (karena income statement di Annual Report sering berada di halaman 15-30), namun membatasi OCR tetap 15 halaman demi efisiensi CPU.

In [21]:
# Setup Google Colab (opsional, jalankan jika di Colab)
try:
    import google.colab
    !apt-get install -y tesseract-ocr
    !pip install pymupdf pytesseract
except ImportError:
    pass

In [22]:
import os
import re
import io
import pandas as pd
import fitz
from PIL import Image
import pytesseract
from tqdm.notebook import tqdm

# Konfigurasi Tesseract OCR untuk lokal Windows
# Jika Anda di Windows, pastikan Tesseract terinstall dan atur path di bawah ini jika diperlukan
tess_paths = [
    r"C:\Program Files\Tesseract-OCR\tesseract.exe",
    r"C:\Program Files (x86)\Tesseract-OCR\tesseract.exe"
]
for p in tess_paths:
    if os.path.exists(p):
        pytesseract.pytesseract.tesseract_cmd = p
        break

In [23]:
def clean_numeric(val_str):
    if not val_str:
        return None
    val_str = val_str.strip()
    is_negative = False
    
    # Deteksi tanda kurung (negatif)
    if val_str.startswith('(') and val_str.endswith(')'):
        is_negative = True
        val_str = val_str[1:-1].strip()
    elif val_str.startswith('-'):
        is_negative = True
        val_str = val_str[1:].strip()
        
    # Bersihkan karakter non-digit
    cleaned = "".join([c for c in val_str if c.isdigit()])
    if not cleaned:
        return 0
        
    val = int(cleaned)
    if is_negative:
        val = -val
    return val

def merge_split_brackets(text):
    """
    Penyatuan tanda kurung terpisah:
    Menyatukan baris tanda kurung negatif ( ) yang terpisah akibat ekstraksi PyMuPDF ke angka di bawahnya.
    """
    lines = text.split('\n')
    cleaned_lines = []
    i = 0
    while i < len(lines):
        line = lines[i].strip()
        if line == '(' or line == ')' or line == '( )':
            j = i + 1
            while j < len(lines) and not lines[j].strip():
                j += 1
            if j < len(lines):
                next_line = lines[j].strip()
                if any(c.isdigit() for c in next_line):
                    if not next_line.startswith('-'):
                        lines[j] = '-' + lines[j]
            i += 1
            continue
        cleaned_lines.append(lines[i])
        i += 1
    return '\n'.join(cleaned_lines)

def extract_text_from_pdf(pdf_path, max_pages=40, max_ocr_pages=15):
    """
    Ekstrak teks dari PDF menggunakan PyMuPDF (fitz).
    Membatasi halaman (misal 40 halaman pertama) karena balance sheet & income statement 
    serinya terletak di bagian awal/tengah laporan keuangan (di Annual Report halaman 15-30).
    """
    doc = fitz.open(pdf_path)
    text = ""
    pages_to_scan = min(len(doc), max_pages)
    
    for i in range(pages_to_scan):
        text += doc[i].get_text()
        
    # Jika teks sangat sedikit, anggap berkas adalah scan dan jalankan OCR fallback
    if len(text.strip()) < 200:
        print(f"Mengaktifkan OCR Fallback untuk berkas: {os.path.basename(pdf_path)}")
        text = ""
        ocr_pages = min(len(doc), max_ocr_pages)
        for i in range(ocr_pages):
            page = doc[i]
            # Render halaman ke gambar
            pix = page.get_pixmap(dpi=150)
            img_data = pix.tobytes("png")
            img = Image.open(io.BytesIO(img_data))
            # OCR dengan bahasa Indonesia dan Inggris
            try:
                page_text = pytesseract.image_to_string(img, lang="ind+eng")
            except Exception as e:
                try:
                    page_text = pytesseract.image_to_string(img, lang="eng")
                except:
                    page_text = ""
            text += page_text + "\n"
            
    doc.close()
    return text

## 1. Pipeline 1: Pemindaian 9 Metrik Lengkap (Original)
Langkah ini mengekstrak seluruh 9 metrik awal dan menyimpannya di `Financial_Metrics_Raw.xlsx`.

In [24]:
# Pola RegEx Indonesia dan Inggris untuk 9 metrik target dengan perbaikan horizontal space [ \t]
patterns = {
    "total_assets": [
        r"(?:jumlah aset|total assets)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)(?!\s*lancar)",
        r"jumlah aset\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "total_liabilities": [
        r"(?:jumlah liabilitas|total liabilities)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)",
        r"jumlah kewajiban\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "total_equity": [
        r"(?:jumlah ekuitas|total equity)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "revenue": [
        r"(?:penjualan dan pendapatan usaha|pendapatan usaha|pendapatan neto|net revenue|revenues)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)",
        r"pendapatan dari kontrak dengan pelanggan\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "net_income": [
        r"(?:jumlah laba \(rugi\)|laba neto tahun berjalan|laba neto yang dapat diatribusikan|laba tahun berjalan|net income)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "accounts_receivable": [
        r"(?:piutang usaha|accounts receivable)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "inventory": [
        r"(?:persediaan|inventories)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "cash": [
        r"(?:kas dan setara kas|cash and cash equivalents)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "cfo": [
        r"(?:arus kas bersih (?:dari|yang diperoleh dari) aktivitas operasi|net cash flows from operating activities)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)",
        r"(?:kas bersih dari aktivitas operasi)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ]
}

def parse_metrics(text):
    result = {}
    text_lower = text.lower()
    
    for key, regex_list in patterns.items():
        result[key] = None
        for pattern in regex_list:
            match = re.search(pattern, text_lower, re.MULTILINE)
            if match:
                val_str = match.group(1)
                result[key] = clean_numeric(val_str)
                break
    return result

In [25]:
# Muat PDF Inventory dan saring untuk kategori Laporan_Keuangan
inventory_path = "PDF_Inventory.xlsx"

if not os.path.exists(inventory_path):
    print(f"ERROR: File {inventory_path} tidak ditemukan! Pastikan telah menyelesaikan Tahap 2A.")
else:
    df_inv = pd.read_excel(inventory_path)
    df_fs = df_inv[df_inv['kategori'] == 'Laporan_Keuangan'].copy()
    
    # Saring berkas pendukung non-utama
    aux_keywords = ['surat', 'pengantar', 'cover', 'checklist', 'pernyataan', 'penjelasan', 'pengungkapan', 'corsec', 'ruis', 'press', 'siaran']
    pattern_aux = '|'.join(aux_keywords)
    df_fs = df_fs[~df_fs['file'].astype(str).str.lower().str.contains(pattern_aux)].copy()
    
    print(f"Total Laporan Keuangan Utama untuk diekstraksi (Pipeline 1): {len(df_fs)}")
    
    extracted_records = []
    
    for idx, row in tqdm(df_fs.iterrows(), total=len(df_fs), desc="Ekstraksi PDF Pipeline 1"):
        pdf_path = row['path']
        kode = row['kode']
        tahun = row['tahun']
        
        if not os.path.exists(pdf_path):
            continue
            
        try:
            text = extract_text_from_pdf(pdf_path, max_pages=15)
            metrics = parse_metrics(text)
            
            record = {
                "kode": kode,
                "tahun": tahun,
                "file": row['file'],
                "path": pdf_path
            }
            record.update(metrics)
            extracted_records.append(record)
            
        except Exception as e:
            print(f"Error memproses {kode} {tahun}: {e}")
            
    df_results = pd.DataFrame(extracted_records)
    df_results.to_excel("Financial_Metrics_Raw.xlsx", index=False)
    print(f"Berhasil menyimpan hasil ekstraksi ke 'Financial_Metrics_Raw.xlsx'. Bentuk: {df_results.shape}")

Total Laporan Keuangan Utama untuk diekstraksi (Pipeline 1): 3936


Ekstraksi PDF Pipeline 1:   0%|          | 0/3936 [00:00<?, ?it/s]

Mengaktifkan OCR Fallback untuk berkas: SPD LK FY 24 ACES.pdf
Mengaktifkan OCR Fallback untuk berkas: SPD LK Adhi 31 Desember 2023.pdf
Mengaktifkan OCR Fallback untuk berkas: SPD_ LK Adhi 31 Desember 2023.pdf
Mengaktifkan OCR Fallback untuk berkas: Cheklist LK PT Polychem Indonesia Tbk 31 Des 2023.pdf
Mengaktifkan OCR Fallback untuk berkas: Perubahan Liabilitas LKT-ADMG-31 Des 2024.pdf
Mengaktifkan OCR Fallback untuk berkas: SPD LKT 2021.pdf
Mengaktifkan OCR Fallback untuk berkas: Perubahan Liabilitas LKT 2022.pdf
Mengaktifkan OCR Fallback untuk berkas: AIMS LK Tahunan per 31 DES 2021 (AUDITED).pdf
Mengaktifkan OCR Fallback untuk berkas: AIMS Perubahan Akun 20pct LK 31 DES 2021 (Audited).pdf
Mengaktifkan OCR Fallback untuk berkas: AIMS LK 31 DES 2022 (audited) Perubahan Akun 20pct.pdf
Mengaktifkan OCR Fallback untuk berkas: AIMS Penyampaikan Kembali LK 31 DES 2023 via XBRL.pdf
Mengaktifkan OCR Fallback untuk berkas: OJK BEI  LKT  2021.pdf
Mengaktifkan OCR Fallback untuk berkas: SPD ALK

## 2. Pipeline 2: Fokus Metrik Beneish M-Score (Deduplicated & Clean)
Langkah ini difokuskan khusus untuk mengekstrak variabel lengkap untuk perhitungan **Beneish M-Score**:
- `total_assets` (Aset)
- `total_liabilities` (Liabilitas) - untuk Leverage Index (LVGI)
- `total_equity` (Ekuitas)
- `current_assets` (Aset Lancar) - untuk Asset Quality Index (AQI)
- `ppe` (Aset Tetap / PP&E) - untuk AQI & Depreciation Index (DEPI)
- `depreciation` (Penyusutan) - untuk DEPI (jika tersedia, jika tidak akan di-default 1.0 pada M-Score)
- `revenue` (Pendapatan/Penjualan) - untuk DSRI, GMI, SGI, SGAI
- `receivables` (Piutang Usaha) - untuk Days Sales in Receivables Index (DSRI)
- `gross_profit` (Laba Kotor) - untuk Gross Margin Index (GMI)
- `selling_expense` (Beban Penjualan) - untuk SG&A Index (SGAI)
- `ga_expense` (Beban Umum & Administrasi) - untuk SG&A Index (SGAI)
- `net_income` (Laba Bersih) - untuk Total Accruals to Total Assets (TATA)
- `cfo` (Arus Kas Aktivitas Operasi) - untuk TATA

**Catatan**: Deduplikasi per (Kode, Tahun) diterapkan di sini dengan memprioritaskan berkas standardized `FinancialStatement` utama.

In [26]:
# Pola RegEx Indonesia dan Inggris lengkap untuk variabel Beneish M-Score
focused_patterns = {
    "total_assets": [
        r"(?:jumlah\s+aset|total\s+assets|jumlah\s+aktiva|total\s+aktiva)\s+(\(?[ \t]*[-\d.,]+[ 	]*\)?)(?!\s*lancar|\s*tidak\s+lancar)",
        r"jumlah\s+aset\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "total_liabilities": [
        r"(?:jumlah\s+liabilitas|total\s+liabilities|jumlah\s+kewajiban)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "total_equity": [
        r"(?:jumlah\s+ekuitas|total\s+equity|jumlah\s+modal)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "current_assets": [
        r"(?:jumlah\s+aset\s+lancar|total\s+current\s+assets|jumlah\s+aktiva\s+lancar)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "ppe": [
        r"(?:aset\s+tetap|property,\s+plant\s+and\s+equipment|property,\s+plant,\s+and\s+equipment|aktiva\s+tetap)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "depreciation": [
        r"(?:penyusutan\s+aset\s+tetap|penyusutan\s+dan\s+amortisasi|depreciation\s+of\s+property|depreciation\s+and\s+amortization)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "revenue": [
        r"(?:penjualan\s+dan\s+pendapatan\s+usaha|pendapatan\s+usaha|pendapatan\s+neto|net\s+revenue|revenues|penjualan\s+bersih|net\s+sales|sales\s+and\s+revenue|pendapatan\s+dari\s+kontrak\s+dengan\s+pelanggan)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "receivables": [
        r"(?:jumlah\s+piutang\s+usaha|piutang\s+usaha|trade\s+receivable|accounts\s+receivable)[^0-9]*?(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "gross_profit": [
        r"(?:jumlah\s+laba\s+bruto|laba\s+bruto|laba\s+kotor|gross\s+profit|total\s+gross\s+profit)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "selling_expense": [
        r"(?:beban\s+penjualan|selling\s+expenses|selling\s+expense)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "ga_expense": [
        r"(?:beban\s+umum\s+dan\s+administrasi|general\s+and\s+administrative\s+expenses|general\s+and\s+administrative\s+expense)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "net_income": [
        r"(?:laba\s+(?:neto|bersih)\s+tahun\s+berjalan|laba\s+neto\s+yang\s+dapat\s+diatribusikan|laba\s+tahun\s+berjalan|net\s+income|laba\s+bersih|laba\s+\(?rugi\)?\s+tahun\s+berjalan|jumlah\s+laba\s+\(?rugi\)?|laba\s+neto)\s+(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ],
    "cfo": [
        r"(?:arus\s+kas\s+bersih[^0-9]*?aktivitas\s+operasi|net\s+cash\s+flow[^0-9]*?operating\s+activit|kas\s+bersih[^0-9]*?aktivitas\s+operasi|net\s+cash\s+provided[^0-9]*?operating\s+activit)[^0-9]*?(\(?[ 	]*[-\d.,]+[ 	]*\)?)"
    ]
}

def parse_focused_metrics(text):
    # Penyatuan bracket
    text_clean = merge_split_brackets(text)
    text_clean = text_clean.lower().replace('|', ' ')
    
    result = {}
    for key, regex_list in focused_patterns.items():
        result[key] = None
        for pattern in regex_list:
            match = re.search(pattern, text_clean, re.MULTILINE)
            if match:
                val_str = match.group(1)
                result[key] = clean_numeric(val_str)
                break
    return result

In [27]:
def filter_unique_financial_statements(df):
    """
    Deduplikasi: Pilih tepat 1 berkas per (Kode, Tahun).
    Memprioritaskan berkas standardized 'FinancialStatement-'.
    """
    unique_records = []
    for (kode, tahun), group in df.groupby(['kode', 'tahun']):
        fs_files = group[group['file'].str.lower().str.contains('financialstatement')]
        if not fs_files.empty:
            unique_records.append(fs_files.iloc[0])
        else: 
            lk_files = group[group['file'].str.lower().str.contains('laporan_keuangan|laporan keuangan|lk_keu|lk')]
            if not lk_files.empty:
                unique_records.append(lk_files.iloc[0])
            else:
                unique_records.append(group.iloc[0])
    return pd.DataFrame(unique_records)

In [28]:
# Eksekusi Pipeline 2 (Fokus Metrik Beneish)
if not os.path.exists(inventory_path):
    print(f"ERROR: File {inventory_path} tidak ditemukan!")
else:
    df_inv = pd.read_excel(inventory_path)
    df_fs = df_inv[df_inv['kategori'] == 'Laporan_Keuangan'].copy()
    
    # Saring berkas pendukung non-utama
    aux_keywords = ['surat', 'pengantar', 'cover', 'checklist', 'pernyataan', 'penjelasan', 'pengungkapan', 'corsec', 'ruis', 'press', 'siaran']
    pattern_aux = '|'.join(aux_keywords)
    df_fs = df_fs[~df_fs['file'].astype(str).str.lower().str.contains(pattern_aux)].copy()
    
    # Terapkan penyaringan unik per (kode, tahun)
    df_fs_unique = filter_unique_financial_statements(df_fs)
    
    print(f"Total Laporan Keuangan Utama untuk diekstraksi (Pipeline 2 - Unik): {len(df_fs_unique)}")
    
    focused_records = []
    
    for idx, row in tqdm(df_fs_unique.iterrows(), total=len(df_fs_unique), desc="Ekstraksi PDF Pipeline 2"):
        pdf_path = row['path']
        kode = row['kode']
        tahun = row['tahun']
        
        if not os.path.exists(pdf_path):
            continue
            
        try:
            # Scan 40 halaman pertama (karena text-based cepat, dan max_ocr_pages tetap 15)
            text = extract_text_from_pdf(pdf_path, max_pages=40, max_ocr_pages=15)
            metrics = parse_focused_metrics(text)
            
            record = {
                "kode": kode,
                "tahun": tahun,
                "file": row['file'],
                "path": pdf_path
            }
            record.update(metrics)
            focused_records.append(record)
            
        except Exception as e:
            print(f"Error memproses {kode} {tahun}: {e}")
            
    df_focused_results = pd.DataFrame(focused_records)
    df_focused_results.to_excel("Financial_Metrics_Focused.xlsx", index=False)
    print(f"Berhasil menyimpan hasil ekstraksi ke 'Financial_Metrics_Focused.xlsx'. Bentuk: {df_focused_results.shape}")

Total Laporan Keuangan Utama untuk diekstraksi (Pipeline 2 - Unik): 2466


Ekstraksi PDF Pipeline 2:   0%|          | 0/2466 [00:00<?, ?it/s]

Berhasil menyimpan hasil ekstraksi ke 'Financial_Metrics_Focused.xlsx'. Bentuk: (2466, 17)
